# Entrenamiento de hoop-ball.tflite (aro y balon)
Entrena un detector EfficientDet-Lite0 con MediaPipe Model Maker sobre datasets de Roboflow y exporta un modelo int8 de menos de 5 MB.

**Nota de versiones:** `mediapipe-model-maker` necesita una version de Python/TensorFlow concreta. Si la celda 2 falla al instalar, usa un entorno de ejecucion de Colab con Python 3.10/3.11 o revisa la version de TensorFlow que pide el paquete.

## 1. Configuracion
Rellena `DATASETS` con el workspace, proyecto y version de Roboflow Universe.

In [ ]:
DATASETS = [
    # {"workspace": "mi-workspace", "project": "mi-proyecto", "version": 1},
]
EPOCAS = 50
BATCH = 16
LR = 0.3
MAX_MB = 5

assert DATASETS, "Rellena DATASETS con al menos un dataset de Roboflow (workspace, project, version)."

## 2. Entorno
Comprueba la GPU, instala dependencias y obtiene `unificar_etiquetas.py` (por `git clone` o subiendolo a mano si el repo es privado).

In [ ]:
import subprocess, sys, os, shutil

try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit(
        "No hay GPU. En Colab: Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > "
        "Acelerador por hardware: GPU T4. Despues vuelve a ejecutar todo."
    )

!pip install -q mediapipe-model-maker roboflow

if not os.path.exists("unificar_etiquetas.py"):
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Ezy01v1/EzyBall", "EzyBall_repo"],
        capture_output=True, text=True,
    )
    origen = os.path.join("EzyBall_repo", "training", "unificar_etiquetas.py")
    if r.returncode == 0 and os.path.exists(origen):
        shutil.copy(origen, "unificar_etiquetas.py")
    else:
        print("No se pudo clonar el repo (puede ser privado). Sube el archivo training/unificar_etiquetas.py:")
        from google.colab import files
        subidos = files.upload()
        nombre = next((n for n in subidos if n.endswith("unificar_etiquetas.py")), None)
        if nombre is None:
            raise SystemExit("No se subio unificar_etiquetas.py; no se puede continuar.")
        if nombre != "unificar_etiquetas.py":
            os.replace(nombre, "unificar_etiquetas.py")

sys.path.insert(0, os.getcwd())
from unificar_etiquetas import unificar_coco, resumen, ErrorDataset
print("Entorno listo.")

## 3. Clave de Roboflow
Se lee del secreto `ROBOFLOW_API_KEY` de Colab; si no existe, se pide por teclado.

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    import getpass
    ROBOFLOW_API_KEY = getpass.getpass("Clave de API de Roboflow: ")
assert ROBOFLOW_API_KEY, "Falta la clave de Roboflow."

## 4. Descarga
Descarga cada dataset en formato COCO.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
descargados = []
for d in DATASETS:
    try:
        proyecto = rf.workspace(d["workspace"]).project(d["project"])
        ds = proyecto.version(d["version"]).download("coco")
    except Exception as e:
        raise SystemExit(f"No se pudo descargar {d}: {e}")
    descargados.append((f"{d['project']}_v{d['version']}", ds.location))
    print("Descargado:", d, "->", ds.location)

## 5. Unificacion
Convierte las etiquetas a `aro`/`balon`, fusiona los datasets (reindexando ids y prefijando nombres de archivo) y deja `datos/<split>/images/` + `datos/<split>/labels.json`, que es lo que pide Model Maker.

In [ ]:
import json, shutil
from pathlib import Path

SPLITS = ["train", "valid", "test"]
shutil.rmtree("datos", ignore_errors=True)

fusion = {}
for split in SPLITS:
    imagenes, anotaciones, categorias = [], [], None
    copias = []
    for prefijo, ruta in descargados:
        carpeta = Path(ruta) / split
        ann_path = carpeta / "_annotations.coco.json"
        if not ann_path.exists():
            print(f"[{prefijo}] sin split '{split}', se omite")
            continue
        coco = json.load(open(ann_path, encoding="utf-8"))
        try:
            coco = unificar_coco(coco)
        except ErrorDataset as e:
            raise SystemExit(f"[{prefijo}/{split}] {e}")
        mapa_id = {}
        for img in coco["images"]:
            nuevo_id = len(imagenes) + 1
            mapa_id[img["id"]] = nuevo_id
            nuevo_nombre = f"{prefijo}__{img['file_name']}"
            copias.append((carpeta / img["file_name"], nuevo_nombre))
            imagenes.append({**img, "id": nuevo_id, "file_name": nuevo_nombre})
        for ann in coco["annotations"]:
            anotaciones.append({**ann, "id": len(anotaciones) + 1, "image_id": mapa_id[ann["image_id"]]})
        categorias = coco["categories"]
    if not imagenes:
        fusion[split] = None
        continue
    destino = Path("datos") / split
    (destino / "images").mkdir(parents=True)
    for origen, nombre in copias:
        shutil.copy(origen, destino / "images" / nombre)
    salida = {"images": imagenes, "categories": categorias, "annotations": anotaciones}
    json.dump(salida, open(destino / "labels.json", "w", encoding="utf-8"), ensure_ascii=False)
    fusion[split] = salida
    print(split, resumen(salida))

assert fusion["train"] and fusion["valid"], "Hacen falta los splits train y valid."
if fusion["test"] is None:
    print("Aviso: no hay split test; se usara valid para evaluar.")

## 6. Entrenamiento
Entrena EfficientDet-Lite0 con los hiperparametros de la celda 1.

In [ ]:
from mediapipe_model_maker import object_detector, quantization

train_data = object_detector.Dataset.from_coco_folder("datos/train", cache_dir="cache/train")
validation_data = object_detector.Dataset.from_coco_folder("datos/valid", cache_dir="cache/valid")
test_dir = "datos/test" if fusion["test"] else "datos/valid"
test_data = object_detector.Dataset.from_coco_folder(test_dir, cache_dir="cache/test")

options = object_detector.ObjectDetectorOptions(
    supported_model=object_detector.SupportedModels.EFFICIENTDET_LITE0,
    hparams=object_detector.HParams(
        epochs=EPOCAS, batch_size=BATCH, learning_rate=LR, export_dir="export"
    ),
)
model = object_detector.ObjectDetector.create(
    train_data=train_data, validation_data=validation_data, options=options
)

## 7. Evaluacion
Imprime el mAP general y el AP por clase sobre el split de test.

In [ ]:
loss, metricas = model.evaluate(test_data, batch_size=BATCH)
print(f"Loss: {loss}")
print("Todas las metricas COCO:")
for k, v in metricas.items():
    print(f"  {k}: {v}")
print()
print("mAP general (AP):", metricas.get("AP"))
for k, v in metricas.items():
    if k.startswith("AP_/") or k.lower() in ("ap_aro", "ap_balon"):
        print("AP por clase", k, ":", v)

## 8. Exportacion int8
Exporta con cuantizacion int8 y comprueba que pesa menos de `MAX_MB`.

In [ ]:
model.export_model(
    "hoop-ball.tflite",
    quantization_config=quantization.QuantizationConfig.for_int8(representative_data=train_data),
)
RUTA_TFLITE = os.path.join("export", "hoop-ball.tflite")
assert os.path.exists(RUTA_TFLITE), f"No se genero {RUTA_TFLITE}; revisa la salida de la celda."
mb = os.path.getsize(RUTA_TFLITE) / 1e6
print(f"Tamano: {mb:.2f} MB")
assert mb < MAX_MB, f"El modelo pesa {mb:.2f} MB y el maximo es {MAX_MB} MB"

## 9. Inspeccion
**Pega la salida de esta celda en la conversacion**: sirve para rellenar el perfil del modelo en la app. Tambien genera `labels.txt` y ejecuta una inferencia de prueba.

In [ ]:
import zipfile
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=RUTA_TFLITE)
interp.allocate_tensors()
entrada = interp.get_input_details()[0]
salidas = interp.get_output_details()

print("===== INSPECCION DEL MODELO =====")
print("Tipo de entrada:", np.dtype(entrada["dtype"]).name)
print("Forma de entrada:", list(entrada["shape"]))
print("Cuantizacion de entrada (escala, punto cero):", entrada["quantization"])
print("Salidas:")
for i, s in enumerate(salidas):
    print(f"  indice {i}: nombre={s['name']} forma={list(s['shape'])} tipo={np.dtype(s['dtype']).name}")

# Etiquetas: Model Maker las incrusta en el .tflite (que es tambien un zip)
etiquetas = None
try:
    with zipfile.ZipFile(RUTA_TFLITE) as z:
        for n in z.namelist():
            if n.endswith(".txt"):
                etiquetas = [l.strip() for l in z.read(n).decode("utf-8").splitlines() if l.strip()]
                break
except zipfile.BadZipFile:
    pass
if not etiquetas:
    print("AVISO: no se encontraron etiquetas en el modelo; se usa el orden por defecto.")
    etiquetas = ["aro", "balon"]
with open("labels.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(etiquetas) + "\n")
print("Orden de etiquetas del modelo:", etiquetas)
print("===== FIN =====")

# Inferencia de prueba con una imagen de test
import glob
from PIL import Image, ImageDraw

rutas = sorted(glob.glob(os.path.join(test_dir, "images", "*")))
if rutas:
    h, w = int(entrada["shape"][1]), int(entrada["shape"][2])
    img = Image.open(rutas[0]).convert("RGB")
    x = np.expand_dims(np.array(img.resize((w, h))), 0).astype(entrada["dtype"])
    interp.set_tensor(entrada["index"], x)
    interp.invoke()
    sal = [interp.get_tensor(s["index"]) for s in salidas]
    print("Formas de la inferencia:", [a.shape for a in sal])
    # Busca cajas [1, N, 4] y puntuaciones [1, N] por forma (el orden varia segun el modelo)
    cajas = next((a for a in sal if a.ndim == 3 and a.shape[-1] == 4), None)
    puntos = next((a for a in sal if a.ndim == 2 and a.shape[0] == 1 and a.dtype.kind == "f"), None)
    clases = next((a for a in sal if a.ndim == 2 and a is not puntos and a.shape[0] == 1), None)
    dib = ImageDraw.Draw(img)
    if cajas is not None and puntos is not None:
        W, H = img.size
        for j in range(cajas.shape[1]):
            if puntos[0, j] < 0.3:
                continue
            y0, x0, y1, x1 = cajas[0, j]  # normalizadas [ymin, xmin, ymax, xmax]
            dib.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline="red", width=3)
            c = int(clases[0, j]) if clases is not None else -1
            dib.text((x0 * W + 4, y0 * H + 4), f"{c}:{puntos[0, j]:.2f}", fill="red")
    else:
        print("No se pudo interpretar la salida automaticamente; revisa las formas impresas arriba.")
    display(img)
else:
    print("No hay imagenes de test para la inferencia de prueba.")

## 10. Descarga
Descarga `hoop-ball.tflite` y `labels.txt`. Copia el `.tflite` a `assets/models/` en la app.

In [ ]:
from google.colab import files
files.download(RUTA_TFLITE)
files.download("labels.txt")